In [1]:
import pandas as pd
import numpy as np
import re
import nltk
import string

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize


In [2]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\home\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\home\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\home\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [3]:
df = pd.read_csv("../data/raw/train.csv")
df.head()


,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\r\nWhy the edits made under my use...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\r\nMore\r\nI can't make any real suggestions...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


In [4]:
label_columns = ["toxic", "severe_toxic", "obscene", 
                 "threat", "insult", "identity_hate"]

df = df[["comment_text"] + label_columns]

df.head()


,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,Explanation\r\nWhy the edits made under my use...,0,0,0,0,0,0
1,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,"""\r\nMore\r\nI can't make any real suggestions...",0,0,0,0,0,0
4,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


In [5]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = str(text).lower()
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    
    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    
    # Remove numbers
    text = re.sub(r'\d+', '', text)
    
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # Tokenize
    tokens = word_tokenize(text)
    
    # Remove stopwords + lemmatize
    tokens = [lemmatizer.lemmatize(word) 
              for word in tokens 
              if word not in stop_words and len(word) > 2]
    
    return " ".join(tokens)


In [6]:
df["cleaned_text"] = df["comment_text"].apply(clean_text)

df.head()


,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate,cleaned_text
0,Explanation\r\nWhy the edits made under my use...,0,0,0,0,0,0,explanation edits made username hardcore metal...
1,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0,daww match background colour seemingly stuck t...
2,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0,hey man really trying edit war guy constantly ...
3,"""\r\nMore\r\nI can't make any real suggestions...",0,0,0,0,0,0,cant make real suggestion improvement wondered...
4,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0,sir hero chance remember page thats


In [7]:
df[["comment_text", "cleaned_text"]].head(10)


,comment_text,cleaned_text
0,Explanation\r\nWhy the edits made under my use...,explanation edits made username hardcore metal...
1,D'aww! He matches this background colour I'm s...,daww match background colour seemingly stuck t...
2,"Hey man, I'm really not trying to edit war. It...",hey man really trying edit war guy constantly ...
3,"""\r\nMore\r\nI can't make any real suggestions...",cant make real suggestion improvement wondered...
4,"You, sir, are my hero. Any chance you remember...",sir hero chance remember page thats
5,"""\r\n\r\nCongratulations from me as well, use ...",congratulation well use tool well talk
6,COCKSUCKER BEFORE YOU PISS AROUND ON MY WORK,cocksucker piss around work
7,Your vandalism to the Matt Shirvington article...,vandalism matt shirvington article reverted pl...
8,Sorry if the word 'nonsense' was offensive to ...,sorry word nonsense offensive anyway intending...
9,alignment on this subject and which are contra...,alignment subject contrary dulithgow


In [8]:
df[df["cleaned_text"] == ""].shape
df = df[df["cleaned_text"] != ""]


In [9]:
df["is_toxic"] = df[label_columns].sum(axis=1) > 0
df["is_toxic"] = df["is_toxic"].astype(int)

df["is_toxic"].value_counts()


is_toxic
0    143264
1     16219
Name: count, dtype: int64

In [10]:
df["cleaned_length"] = df["cleaned_text"].apply(len)
df["cleaned_length"].describe()


count    159483.000000
mean        242.408664
std         377.501794
min           2.000000
25%          56.000000
50%         124.000000
75%         265.000000
max        5000.000000
Name: cleaned_length, dtype: float64

In [11]:
df.to_csv("../data/processed/cleaned_comments.csv", index=False)

print("✅ Cleaned dataset saved successfully!")


✅ Cleaned dataset saved successfully!
